In [ ]:
# 数据前处理

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import torch
import pickle

# 文件路径
TRAIN_EXCEL_FILE = 'train_set_L_CCS.xlsx'

# 从 Excel 文件读取数据（第一个工作表用于训练）
df_train = pd.read_excel(TRAIN_EXCEL_FILE, sheet_name=0, usecols=[0, 1, 2], header=0)
df_train.columns = ['Sequence', 'Numerical_Feature', 'y']

# 从 Excel 文件读取数据（第二个工作表用于预测，只有序列列）
df_predict = pd.read_excel(TRAIN_EXCEL_FILE, sheet_name=1, usecols=[0], header=0)
df_predict.columns = ['Sequence']

# 检查数据
print("训练数据前几行:")
print(df_train.head())
print("\n预测数据前几行:")
print(df_predict.head())

# 分离训练特征和目标变量
X_seq = df_train['Sequence'].values.astype(str)  # 序列特征
X_num = df_train['Numerical_Feature'].values.astype(float).reshape(-1, 1)  # 数值特征
y = df_train['y'].values.astype(float)  # 目标变量（回归）

# 划分训练集和验证集
X_seq_train, X_seq_val, X_num_train, X_num_val, y_train, y_val = train_test_split(
    X_seq, X_num, y, test_size=0.2, random_state=42
)

# 标准化数值特征
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_val_scaled = scaler.transform(X_num_val)

# 确定最大序列长度（用于One-Hot编码）
max_length = max(max(len(seq) for seq in X_seq_train), max(len(seq) for seq in X_seq_val))

# One-Hot编码序列特征
def one_hot_encode_sequences(sequences, max_length):
    # 创建氨基酸词典（包含所有可能的氨基酸）
    amino_acids = ['A', 'R', 'N', 'D', 'C', 'E', 'Q', 'G', 'H', 'I', 
                   'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V', 'X']  # 'X' 用于填充
    
    # 初始化编码器
    encoder = OneHotEncoder(categories=[amino_acids], sparse_output=False)
    
    # 编码序列
    encoded_sequences = []
    for seq in sequences:
        # 截断或填充序列到固定长度
        seq = seq.ljust(max_length, 'X')[:max_length]
        # 转换为字符列表
        seq_chars = list(seq)
        # 编码
        encoded = encoder.fit_transform(np.array(seq_chars).reshape(-1, 1))
        encoded_sequences.append(encoded)
    
    return np.array(encoded_sequences), encoder, max_length

# 对训练集和验证集进行One-Hot编码
X_seq_train_encoded, encoder, max_length = one_hot_encode_sequences(X_seq_train, max_length)
X_seq_val_encoded, _, _ = one_hot_encode_sequences(X_seq_val, max_length)

# 打印分割后的数据集大小
print("\n训练集大小:")
print(f"序列特征: {X_seq_train_encoded.shape}")
print(f"数值特征: {X_num_train_scaled.shape}")
print(f"目标变量: {y_train.shape}")

print("\n验证集大小:")
print(f"序列特征: {X_seq_val_encoded.shape}")
print(f"数值特征: {X_num_val_scaled.shape}")
print(f"目标变量: {y_val.shape}")

# 对预测数据进行预处理
X_seq_predict = df_predict['Sequence'].values.astype(str)

# 使用训练数据的编码器和最大长度对预测数据进行编码
def encode_new_sequences(sequences, encoder, max_length):
    encoded_sequences = []
    for seq in sequences:
        # 截断或填充序列到固定长度
        seq = seq.ljust(max_length, 'X')[:max_length]
        # 转换为字符列表
        seq_chars = list(seq)
        # 使用训练好的编码器进行编码
        encoded = encoder.transform(np.array(seq_chars).reshape(-1, 1))
        encoded_sequences.append(encoded)
    
    return np.array(encoded_sequences)

# 对预测数据进行编码
X_seq_predict_encoded = encode_new_sequences(X_seq_predict, encoder, max_length)

# 为预测数据创建数值特征（假设为0，因为预测数据中没有数值特征）
X_num_predict = np.zeros((len(X_seq_predict), 1))
X_num_predict_scaled = scaler.transform(X_num_predict)  # 使用训练数据的标准化器

print("\n预测数据大小:")
print(f"序列特征: {X_seq_predict_encoded.shape}")
print(f"数值特征: {X_num_predict_scaled.shape}")

# 保存预处理的模型和数据（用于后续预测）
with open('model_preprocessing.pkl', 'wb') as f:
    pickle.dump({
        'encoder': encoder,
        'scaler': scaler,
        'max_length': max_length,
        'X_seq_predict_encoded': X_seq_predict_encoded,
        'X_num_predict_scaled': X_num_predict_scaled
    }, f)

# 保存训练数据和验证数据（用于模型训练）
with open('training_data.pkl', 'wb') as f:
    pickle.dump({
        'X_seq_train_encoded': X_seq_train_encoded,
        'X_num_train_scaled': X_num_train_scaled,
        'y_train': y_train,
        'X_seq_val_encoded': X_seq_val_encoded,
        'X_num_val_scaled': X_num_val_scaled,
        'y_val': y_val
    }, f)

print("\n数据预处理完成，模型训练数据和预测数据已保存。")

In [ ]:
# One-hot 编码器
from sklearn.preprocessing import OneHotEncoder

def preprocess(X, max_length, padding_token="Z"):
    padded_sequences = [seq.ljust(max_length, padding_token) for seq in X]
    sequences_array = np.array([list(seq) for seq in padded_sequences])
    encoder = OneHotEncoder(sparse_output=False, dtype=int, handle_unknown="ignore")
    X_encoded = encoder.fit_transform(sequences_array.reshape(-1, 1))
    X_encoded = X_encoded.reshape(len(X), max_length, -1)
    return X_encoded, encoder

max_length = max(max(len(seq) for seq in X_seq_train),
                 max(len(seq) for seq in X_seq_val))

X_train_encoded, encoder = preprocess(X_seq_train, max_length)
X_val_encoded, _ = preprocess(X_seq_val, max_length)

print(max_length)
print(X_train_encoded.shape)
print(X_val_encoded.shape)

In [ ]:
# 将预处理后的数据保存到文件中
np.save('X_train_encoded.npy', X_train_encoded)
np.save('X_num_train_encoded.npy', X_num_train_scaled)
np.save('y_train.npy', y_train)
np.save('X_val_encoded.npy', X_val_encoded)
np.save('X_num_val_encoded.npy', X_num_val_scaled)
np.save('y_val.npy', y_val)

print(f"训练集样本数: {len(X_seq_train)}")
print(f"验证集样本数: {len(X_seq_val)}")

In [ ]:
# 打印类别和对应的 One-Hot 编码
categories = encoder.categories_[0]  # 获取所有类别（字符）
n_categories = len(categories)       # 类别数量
print(f"Total categories: {n_categories}")
print("Character to One-Hot Mapping:")

for i, char in enumerate(categories):
    one_hot_vector = np.zeros(n_categories, dtype=int)  # 创建全零向量
    one_hot_vector[i] = 1                              # 设置当前位置为 1
    print(f"'{char}': {one_hot_vector}")

In [ ]:
print(X_seq_train[0])
X_train_encoded = np.load('X_train_encoded.npy')
np.savetxt('x0.txt', X_train_encoded[0])

In [28]:
# 以下为训练过程
# library and Dataset

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sklearn.utils.class_weight import compute_class_weight

class PeptideDataset(Dataset):
    def __init__(self, X_file, X_num_file, y_file):
        self.X = np.load(X_file)
        self.X_num = np.load(X_num_file)
        self.y = np.load(y_file)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32), 
            torch.tensor(self.X_num[idx], dtype=torch.float32), 
            torch.tensor(self.y[idx], dtype=torch.float32)
        )
    


In [29]:
from torch.utils.data import DataLoader

# 加载数据集
train_dataset = PeptideDataset('X_train_encoded.npy', 'X_num_train_encoded.npy', 'y_train.npy')
val_dataset = PeptideDataset('X_val_encoded.npy', 'X_num_val_encoded.npy','y_val.npy')

# 定义 DataLoader
batch_size = 2048  # 批量大小
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [30]:
# 主程序

d_model = 128    # 这应该与 Transformer 的定义一致
learning_rate = 0.005
weight_decay = 0.0002
patience = 30

cnn_sequence_size = max_length

In [31]:
# 位置编码

import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)

    def forward(self, x):
        x = x + self.encoding[:, :x.size(1), :].to(x.device)
        return x

In [32]:
import lightning.pytorch as pl
from torch.optim.lr_scheduler import ReduceLROnPlateau
from scipy.stats import pearsonr  # 虽然未使用，但保留导入不影响

class EnsembleClassifier(pl.LightningModule):
    def __init__(self,
                 input_size, numerical_feature_size, learning_rate, weight_decay,
                 trans_num_layers=2, trans_d_model=96, trans_nhead=16, trans_dim_feedforward=128, trans_dropout=0.2,
                 cnn_out_channels=64, cnn_sequence_size=100, kernel_size1=(55, 21), kernel_size2=(3, 1),
                 lstm_hidden_size=128, lstm_num_layers=2, bidirectional=False,
                 dense_hidden_size=128, dense_chg_hidden_size=8,
                 ):
        super(EnsembleClassifier, self).__init__()
        self.save_hyperparameters()
        self.numerical_feature_size = numerical_feature_size
        # 模型结构定义（与原代码一致，未修改）
        self.embedding = nn.Linear(input_size, trans_d_model)
        self.position_encoding = PositionalEncoding(trans_d_model)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=trans_d_model, nhead=trans_nhead, dim_feedforward=trans_dim_feedforward, dropout=trans_dropout
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=trans_num_layers)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, cnn_out_channels, kernel_size=kernel_size1, padding=(kernel_size1[0]//2, 0)),
            nn.BatchNorm2d(cnn_out_channels),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(1, 1)),
            nn.Conv2d(cnn_out_channels, int(cnn_out_channels / 2), kernel_size=kernel_size2, padding=(1, 0)),
            nn.BatchNorm2d(int(cnn_out_channels / 2)),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
            nn.Flatten()
        )
        self.lstm = nn.LSTM(input_size, lstm_hidden_size, lstm_num_layers, batch_first=True, bidirectional=bidirectional)
        self.fc_chg = nn.Linear(numerical_feature_size, dense_chg_hidden_size)
        self.fc_chg_activation = nn.ELU()
        transformer_output_dim = trans_d_model
        cnn_output_dim = self._get_cnn_output_dim()
        lstm_output_dim = lstm_hidden_size * self.hparams.cnn_sequence_size * (2 if bidirectional else 1)
        chg_output_dim = dense_chg_hidden_size
        self.fc = nn.Linear(transformer_output_dim + cnn_output_dim + lstm_output_dim + chg_output_dim, dense_hidden_size)
        self.fc_activation = nn.ELU()
        self.fc2 = nn.Linear(dense_hidden_size, 1)
        self.criterion = nn.MSELoss()

    def _get_cnn_output_dim(self):
        x = torch.randn(1, 1, self.hparams.cnn_sequence_size, self.hparams.input_size)
        return self.cnn(x).numel()

    def forward(self, x_encoded, x_numerical):
        # 前向传播逻辑（与原代码一致，未修改）
        x_transformer = self.embedding(x_encoded)
        x_transformer = self.position_encoding(x_transformer)
        x_transformer = x_transformer.permute(1, 0, 2)
        x_transformer = self.transformer_encoder(x_transformer)
        x_transformer = x_transformer.mean(dim=0)
        x_cnn = x_encoded.unsqueeze(1)
        x_cnn = self.cnn(x_cnn)
        output, _ = self.lstm(x_encoded)
        x_lstm = output.reshape(output.size(0), -1)
        x_chg = self.fc_chg(x_numerical)
        x_chg = self.fc_chg_activation(x_chg)
        x_combined = torch.cat((x_transformer, x_cnn, x_lstm, x_chg), dim=1)
        out = self.fc(x_combined)
        out = self.fc_activation(out)
        out = self.fc2(out)
        return out

    def training_step(self, batch, batch_idx):
        x_encoded, x_numerical, y = batch
        x_encoded, x_numerical, y = x_encoded.to(self.device), x_numerical.to(self.device), y.to(self.device)
        y = y.view(-1, 1)
        outputs = self(x_encoded, x_numerical)
        loss = self.criterion(outputs, y)
        
        mae = torch.abs(outputs - y) 
        avg_mae = torch.mean(mae)
        relative_mae = (mae / torch.abs(y)).mean()
    
        # 决定系数 R²
        total_variance = torch.sum((y - torch.mean(y)) ** 2)
        explained_variance = torch.sum((outputs - y) ** 2)
        r2 = 1 - explained_variance / total_variance  # 保留R²计算，移除皮尔逊
    
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_mae', avg_mae, prog_bar=True)
        self.log('train_rmae', relative_mae, prog_bar=True)
        self.log('train_r2', r2, prog_bar=True)
        # 移除了 self.log("train_pearson", pearson_corr, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        x_encoded, x_numerical, y = batch
        x_encoded, x_numerical, y = x_encoded.to(self.device), x_numerical.to(self.device), y.to(self.device)
        y = y.view(-1, 1)
        outputs = self(x_encoded, x_numerical)
        loss = self.criterion(outputs, y)
        mae = torch.abs(outputs - y) 
        avg_mae = torch.mean(mae)
        relative_mae = (mae / torch.abs(y)).mean()
    
        # 决定系数 R²
        total_variance = torch.sum((y - torch.mean(y)) ** 2)
        explained_variance = torch.sum((outputs - y) ** 2)
        r2 = 1 - explained_variance / total_variance  # 保留R²计算，移除皮尔逊
    
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_mae', avg_mae, prog_bar=True)
        self.log('val_rmae', relative_mae, prog_bar=True)
        self.log('val_r2', r2, prog_bar=True)
        # 移除了 self.log("val_pearson", pearson_corr, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x_encoded, x_numerical, y = batch
        x_encoded, x_numerical, y = x_encoded.to(self.device), x_numerical.to(self.device), y.to(self.device)
        y = y.view(-1, 1)
        outputs = self(x_encoded, x_numerical)
        loss = self.criterion(outputs, y)
        mae = torch.abs(outputs - y) 
        avg_mae = torch.mean(mae)
        relative_mae = (mae / torch.abs(y)).mean()
    
        # 决定系数 R²
        total_variance = torch.sum((y - torch.mean(y)) ** 2)
        explained_variance = torch.sum((outputs - y) ** 2)
        r2 = 1 - explained_variance / total_variance  # 保留R²计算，移除皮尔逊
    
        self.log('test_loss', loss, prog_bar=True)
        self.log('test_mae', avg_mae, prog_bar=True)
        self.log('test_rmae', relative_mae, prog_bar=True)
        self.log('test_r2', r2, prog_bar=True)
        # 移除了 self.log("test_pearson", pearson_corr, prog_bar=True)
        # 保存预测值和真值（若不需要可删除）
        self.test_preds.append(outputs.cpu())
        self.test_targets.append(y.cpu())

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=self.hparams.weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

    def on_test_epoch_start(self):
        self.test_preds = []
        self.test_targets = []

    def on_test_epoch_end(self):
        self.test_preds = torch.cat(self.test_preds)
        self.test_targets = torch.cat(self.test_targets)

In [ ]:
# 训练

from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, RichProgressBar

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='checkpoints/',
    filename='best-checkpoint-{epoch:02d}-{val_loss:.2f}',
    save_top_k=1,
    mode='min'
)

rich_progress = RichProgressBar()
early_stopping = pl.callbacks.EarlyStopping(monitor='val_loss', patience=patience, mode='min')

# 获取 one-hot 编码后的特征维度
input_size = X_train_encoded.shape[-1]
numerical_feature_size = X_num_train_scaled.shape[-1] # 数值特征的维度

model = EnsembleClassifier(
    input_size=input_size,
    numerical_feature_size=numerical_feature_size,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    cnn_sequence_size=max_length,
)

rich_progress = RichProgressBar()
trainer = pl.Trainer(
    max_epochs=1000,
    callbacks=[early_stopping, rich_progress, checkpoint_callback],
    accelerator='auto',
    devices=1,
)


trainer.fit(model, train_loader, val_loader)

In [ ]:
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl

import pickle

# ---------------------------
# 1. 加载模型和预处理工具
# ---------------------------
# 加载模型
best_model_path = "checkpoints/L-CCS.ckpt"
model = EnsembleClassifier.load_from_checkpoint(
    best_model_path,
    map_location=torch.device("cpu"),  # 如需 GPU 改为 "cuda"
)
model.eval()

# 加载预处理工具和数据
with open('model_preprocessing.pkl', 'rb') as f:
    preprocessing = pickle.load(f)

# 提取预处理工具和数据
encoder = preprocessing['encoder']
scaler = preprocessing['scaler']
max_length = preprocessing['max_length']
X_seq_predict_encoded = preprocessing['X_seq_predict_encoded']
X_num_predict_scaled = preprocessing['X_num_predict_scaled']

# 加载原始预测数据（用于保存结果）
df_predict = pd.read_excel('train_set_L_CCS.xlsx', sheet_name=1, usecols=[0], header=0)
df_predict.columns = ['Sequence']

print(f"加载了 {len(df_predict)} 条预测数据")


# ---------------------------
# 2. 模型预测（支持大批次分块处理）
# ---------------------------
def predict_in_batches(model, x_seq, x_num, batch_size=32):
    """分批次预测，避免内存溢出"""
    model.eval()
    predictions = []
    device = next(model.parameters()).device  # 获取模型所在设备
    
    for i in range(0, len(x_seq), batch_size):
        # 取批次数据
        batch_seq = torch.from_numpy(x_seq[i:i+batch_size]).float().to(device)
        batch_num = torch.from_numpy(x_num[i:i+batch_size]).float().to(device)
        
        # 预测
        with torch.no_grad():
            y_hat = model(batch_seq, batch_num)
            predictions.extend(y_hat.cpu().numpy())
    
    return np.array(predictions).flatten()

# 执行预测
predictions = predict_in_batches(
    model,
    X_seq_predict_encoded,
    X_num_predict_scaled,
    batch_size=32  # 根据内存调整
)


# ---------------------------
# 3. 保存预测结果到 Excel
# ---------------------------
df_predict['Prediction'] = predictions
output_path = "prediction_results.xlsx"
df_predict.to_excel(output_path, index=False)

print(f"预测完成！结果已保存至 {output_path}")
print(f"前5条预测结果：\n{df_predict.head()}")